## Synthetic Dataset Creation for Experimental Validation

In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta
import numpy as np

# 1. Project Setup
total_users = 50000
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 6, 30)

countries = ['USA', 'India', 'UK', 'Germany', 'Australia', 'Canada']
devices = ['Desktop', 'Mobile', 'Tablet']
groups = ['Control', 'Variant']

# SaaS subscription price
sub_price = 29.99 

data = []

print(f"Engineering longitudinal SaaS dataset for {total_users} users...")

for i in range(1, total_users + 1):
    user_id = f"user_{i}"
    country = random.choice(countries)
    device = random.choice(devices)
    
    # Step 5: Assign A/B Test Group
    group = random.choice(groups)
    
    # Random initial visit date within the 6 month window
    days_range = (end_date - start_date).days
    visit_time = start_date + timedelta(days=random.randint(0, days_range), hours=random.randint(0, 23))
    
    # 1. Funnel Step: Visit Website (Everyone does this)
    data.append([user_id, 'visit_website', visit_time.strftime('%Y-%m-%d %H:%M:%S'), device, country, group, 0.0])
    
    # Set conversion probabilities based on Experiment Group
    # The Variant group has slightly better UX, so they convert higher!
    if group == 'Control':
        signup_prob, onboard_prob, sub_prob = 0.35, 0.60, 0.25
    else: # Variant
        signup_prob, onboard_prob, sub_prob = 0.42, 0.68, 0.35

    # 2. Funnel Step: Sign Up
    if random.random() < signup_prob:
        signup_time = visit_time + timedelta(minutes=random.randint(2, 15))
        data.append([user_id, 'sign_up', signup_time.strftime('%Y-%m-%d %H:%M:%S'), device, country, group, 0.0])
        
        # 3. Funnel Step: Complete Onboarding
        if random.random() < onboard_prob:
            onboard_time = signup_time + timedelta(hours=random.randint(1, 48))
            data.append([user_id, 'complete_onboarding', onboard_time.strftime('%Y-%m-%d %H:%M:%S'), device, country, group, 0.0])
            
            # 4. Funnel Step: Subscribe (Generates Initial Revenue)
            if random.random() < sub_prob:
                sub_time = onboard_time + timedelta(days=random.randint(0, 3), hours=random.randint(1, 12))
                data.append([user_id, 'subscribe', sub_time.strftime('%Y-%m-%d %H:%M:%S'), device, country, group, sub_price])
                
                # Step 3: Cohort Retention (Renewals over the next 3 months)
                # If they subscribed early enough in the year, do they stick around?
                for month in range(1, 4):
                    renewal_time = sub_time + timedelta(days=30 * month)
                    
                    # Cut off events that happen after our June 30th end date
                    if renewal_time > end_date:
                        break
                        
                    # Drop-off logic: Month 1 retention is high, Month 3 is lower
                    retention_chance = 0.70 if month == 1 else (0.50 if month == 2 else 0.30)
                    
                    if random.random() < retention_chance:
                        data.append([user_id, 'renew', renewal_time.strftime('%Y-%m-%d %H:%M:%S'), device, country, group, sub_price])
                    else:
                        break # They churned, stop generating future renewals

# Create DataFrame
columns = ['user_id', 'event_name', 'timestamp', 'device', 'country', 'experiment_group', 'revenue']
df = pd.DataFrame(data, columns=columns)

# Sort by UserID and Timestamp to make it perfectly sequential
df = df.sort_values(by=['user_id', 'timestamp'])

# Save to CSV
file_name = 'saas_product_events_200k.csv'
df.to_csv(file_name, index=False)

print(f"✅ Success! Generated {len(df)} total events.")
print(f"💾 Saved strictly formatted dataset to: {file_name}")

Engineering longitudinal SaaS dataset for 50000 users...
✅ Success! Generated 88488 total events.
💾 Saved strictly formatted dataset to: saas_product_events_200k.csv


## A/B Test Performance Analysis & Revenue Impact Assessment

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Load Data
df = pd.read_csv('ab_test_result.csv')

# Split the groups
control = df[df['experiment_group'] == 'Control']
variant = df[df['experiment_group'] == 'Variant']

# Extract metrics
control_rev = control['total_user_revenue']
variant_rev = variant['total_user_revenue']

control_retained = control['is_retained']
variant_retained = variant['is_retained']

# 2. Conversion & Retention Rates
control_conv = (control_rev > 0).mean() * 100
variant_conv = (variant_rev > 0).mean() * 100

control_ret_rate = control_retained.mean() * 100
variant_ret_rate = variant_retained.mean() * 100

# 3. Revenue Uplift
control_mean = control_rev.mean()
variant_mean = variant_rev.mean()
revenue_uplift = ((variant_mean - control_mean) / control_mean) * 100

# 4. P-Value (Welch's T-Test for Revenue)
t_stat, p_value = stats.ttest_ind(variant_rev, control_rev, equal_var=False)

# 5. Confidence Interval (95%) for Revenue
mean_diff = variant_mean - control_mean
se_diff = np.sqrt(variant_rev.var()/len(variant_rev) + control_rev.var()/len(control_rev))
margin_of_error = 1.96 * se_diff
ci_lower = mean_diff - margin_of_error
ci_upper = mean_diff + margin_of_error

# Output the complete Framework
print("--- A/B Test Framework Results ---")
print(f"Control Conversion Rate (Made a purchase): {control_conv:.2f}%")
print(f"Variant Conversion Rate (Made a purchase): {variant_conv:.2f}%\n")

print(f"Control Retention Rate (Renewed): {control_ret_rate:.2f}%")
print(f"Variant Retention Rate (Renewed): {variant_ret_rate:.2f}%\n")

print(f"Control ARPU (Revenue): ${control_mean:.2f}")
print(f"Variant ARPU (Revenue): ${variant_mean:.2f}")
print(f"Revenue Uplift: {revenue_uplift:.2f}%\n")

print(f"P-Value (Revenue): {p_value:.5f}")
print(f"95% Confidence Interval for ARPU Difference: [${ci_lower:.2f}, ${ci_upper:.2f}]\n")

# 6. Recommendation Engine
if p_value < 0.05 and mean_diff > 0:
    print("Conclusion: The Variant feature caused a statistically significant increase in revenue.")
    print(f"We can be 95% confident that the new feature generates between ${ci_lower:.2f} and ${ci_upper:.2f} more per user. SHIP IT! 🚀")
else:
    print("Conclusion: The Variant did not show a statistically significant positive impact. Do not ship.")

--- A/B Test Framework Results ---
Control Conversion Rate (Made a purchase): 5.16%
Variant Conversion Rate (Made a purchase): 10.01%

Control Retention Rate (Renewed): 3.06%
Variant Retention Rate (Renewed): 5.79%

Control ARPU (Revenue): $2.89
Variant ARPU (Revenue): $5.58
Revenue Uplift: 92.87%

P-Value (Revenue): 0.00000
95% Confidence Interval for ARPU Difference: [$2.40, $2.97]

Conclusion: The Variant feature caused a statistically significant increase in revenue.
We can be 95% confident that the new feature generates between $2.40 and $2.97 more per user. SHIP IT! 🚀
